# **TaniMol: 04 - Butina Clustering**

The primary objective is to identify high-density chemical scaffolds and distinct chemotypes within the dataset. The Butina algorithm, designed specifically for cheminformatics, groups molecules based on a defined Tanimoto similarity threshold. It preserves chemically unique outliers as 'singletons' and assigns actual molecular structures as cluster centroids.

**Input:** Preprocessed activity data and Tanimoto similarity `.npy` matrices   
**Output:** Cluster dictionaries mapping centroids to their members, and singletons list

In [1]:
import numpy as np
import pandas as pd

from src.config import PROCESSED_DIR, MORGAN_SIM_PATH, MACCS_SIM_PATH, RDKIT_SIM_PATH
from src.clustering import cluster_similarity_matrix, analyze_clusters

### **1. Load Preprocessed Data and Matrices**

Load the standardized biological activity dataset and the precomputed pairwise Tanimoto similarity matrices. These matrices represent the complete structural relationship network for all compounds.


In [2]:
df = pd.read_csv(PROCESSED_DIR / "cleaned_activities.csv")
print(f"Loaded {len(df)} molecules with bioactivity data.")

print("Loading Tanimoto Matrices...")
morgan_sim = np.load(MORGAN_SIM_PATH)
print(f"\n[Morgan] Done. Matrix shape: {morgan_sim.shape}")

maccs_sim = np.load(MACCS_SIM_PATH)
print(f"[MACCS] Done. Matrix shape: {maccs_sim.shape}")

rdkit_sim = np.load(RDKIT_SIM_PATH)
print(f"[RDKit] Done. Matrix shape: {rdkit_sim.shape}")

Loaded 11012 molecules with bioactivity data.
Loading Tanimoto Matrices...

[Morgan] Done. Matrix shape: (11012, 11012)
[MACCS] Done. Matrix shape: (11012, 11012)
[RDKit] Done. Matrix shape: (11012, 11012)


### **2. Implementation of Butina Clustering**

Apply the Butina algorithm across the three diverse molecular representations. A strict Tanimoto similarity cutoff (`THRESHOLD = 0.6`) is enforced to ensure that clustered molecules share a significant proportion of their topological features.

*Threshold value can be adjusted in the cell below*


In [3]:
THRESHOLD = 0.6  # Standard similarity cutoff 

print(f"Clustering Morgan fingerprints (Tanimoto Threshold: {THRESHOLD})...")
morgan_clusters, morgan_singletons = cluster_similarity_matrix(morgan_sim, THRESHOLD)
print(f"Found {len(morgan_clusters)} clusters and {len(morgan_singletons)} singletons.")

print(f"\nClustering MACCS keys (Tanimoto Threshold: {THRESHOLD})...")
maccs_clusters, maccs_singletons = cluster_similarity_matrix(maccs_sim, THRESHOLD)
print(f"Found {len(maccs_clusters)} clusters and {len(maccs_singletons)} singletons.")

print(f"\nClustering RDKit fingerprints (Tanimoto Threshold: {THRESHOLD})...")
rdkit_clusters, rdkit_singletons = cluster_similarity_matrix(rdkit_sim, THRESHOLD)
print(f"Found {len(rdkit_clusters)} clusters and {len(rdkit_singletons)} singletons.")

Clustering Morgan fingerprints (Tanimoto Threshold: 0.6)...
Found 497 clusters and 430 singletons.

Clustering MACCS keys (Tanimoto Threshold: 0.6)...
Found 28 clusters and 10 singletons.

Clustering RDKit fingerprints (Tanimoto Threshold: 0.6)...
Found 146 clusters and 126 singletons.


### **3. Cluster Analysis**

Extracting key structural statistics from the clustering output provides insight into the chemical diversity of the dataset.

- **Total Clusters:** The number of distinct chemical scaffolds (core structures) identified.
- **Singletons:** Unique molecules that do not share structural similarity with any other compound in the dataset (novel chemotypes).
- **Biggest Cluster Size:** High values indicate extensively explored chemical series, often representing a heavily patented or optimized scaffold.
- **Large Clusters (>50):** Demonstrates the number of major chemical families dominating the dataset.


#### **A. Morgan Fingerprints (ECFP4)**

In [4]:
morgan_stats = analyze_clusters(morgan_clusters, morgan_singletons)


--- Clustering Analysis ---
Total Clusters:           497
Total Singletons:         430
Molecules in Clusters:    10582
Biggest Cluster Size:     821 molecules
Clusters > 50 molecules:  47
---------------------------



#### **B. MACCS Keys**

In [5]:
maccs_stats = analyze_clusters(maccs_clusters, maccs_singletons)


--- Clustering Analysis ---
Total Clusters:           28
Total Singletons:         10
Molecules in Clusters:    11002
Biggest Cluster Size:     5317 molecules
Clusters > 50 molecules:  10
---------------------------



#### **C. RDKit Topological Fingerprints**


In [6]:
rdkit_stats = analyze_clusters(rdkit_clusters, rdkit_singletons)


--- Clustering Analysis ---
Total Clusters:           146
Total Singletons:         126
Molecules in Clusters:    10886
Biggest Cluster Size:     1659 molecules
Clusters > 50 molecules:  21
---------------------------

